# NOPP-Aleutians Wirewalker RBR Concerto CTD — processing

Trimmed view of the processing products (L1 → L2 → L3 → L3i), ending at the L3 interpolated sections. Derived from `wirewalker_ctd_plots.ipynb`.


## 0. Build the products (run the processing pipeline)

This notebook is a convenient front-end to `process_wirewalker_rbr.py`. The cell below (re)builds the **L1 → L2 → L3** NetCDF products from the raw `.rsk` named in `config.json`, then the sections below load and plot them.

- Set `RUN_PROCESSING = False` to skip the build and just (re)load existing products.
- Set `MAX_CASTS` (e.g. `30`) for a quick test on the first N casts.
- Use `LEVELS` to rebuild only some levels (`'all'`, or e.g. `'23'`).

In [ ]:
# === Build the products: run the L1 -> L2 -> L3 processing pipeline ===
# This notebook is a front-end to process_wirewalker_rbr.py. The cell below rebuilds
# the NetCDF products from the raw .rsk named in config.json; the cells further down
# load and plot them. Point at a different deployment by editing config.json
# (rsk_file, output_dir, basename, metadata) -- no paths are hardcoded here.
import sys, json
from pathlib import Path

def _find_repo(start):
    for d in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (d / 'config.json').exists():
            return d
    raise FileNotFoundError(f'config.json not found at/above {start}')

REPO = _find_repo(Path.cwd())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import process_wirewalker_rbr as wwp

# Release any product files this kernel still has open from a previous run, so the
# build can overwrite them (an open xarray handle locks the .nc -> PermissionError).
for _v in ('L1', 'L2', 'L2t', 'L3', 'L3i'):
    if _v in globals():
        try: globals()[_v].close()
        except Exception: pass
        globals().pop(_v, None)

# ----- controls -----
RUN_PROCESSING = True     # False -> skip the build and just (re)load existing products below
LEVELS         = 'all'    # 'all', or any of '1'/'2'/'3' (e.g. '23' = rebuild L2 then L3)
MAX_CASTS      = None      # e.g. 30 for a quick test on the first N casts; None = full deployment

# use this repo's config.json (rsk path, output dir, deployment metadata)
wwp.apply_config(wwp.load_config(REPO / 'config.json'))
print(f'[config] {wwp.MOORING} | rsk={wwp.RSK_PATH.name} | out={wwp.OUTPUT_DIR}')

if RUN_PROCESSING:
    if LEVELS == 'all' or '1' in LEVELS: wwp.build_L1(MAX_CASTS)
    if LEVELS == 'all' or '2' in LEVELS: wwp.build_L2(MAX_CASTS)
    if LEVELS == 'all' or '3' in LEVELS: wwp.build_L3()
    print('[done] processing complete')
else:
    print('RUN_PROCESSING = False -> skipping build; loading existing products below.')

In [ ]:
import os
import time as _time
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import gsw
import json
from pathlib import Path

# Locate the repo (the folder containing config.json) by walking up from the working dir,
# then read paths from the config -- no hardcoded machine paths.
def _find_repo(start):
    for d in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (d / 'config.json').exists():
            return d
    raise FileNotFoundError(f'config.json not found at/above {start}')

REPO = _find_repo(Path.cwd())
CFG  = json.load(open(REPO / 'config.json'))
OUT  = Path(os.environ.get('WW_OUTPUT_DIR', CFG['output_dir'])).expanduser()  # data products
BASE = CFG['basename']
FIGS = REPO / 'figs'; FIGS.mkdir(exist_ok=True)                              # figures stay in the repo

L1_FILE  = OUT / 'L1' / f'{BASE}_L1_converted.nc'
L2_FILE  = OUT / 'L2' / f'{BASE}_L2_upcast_grid0.5m.nc'
L3_FILE  = OUT / 'L3' / f'{BASE}_L3_grid1m_30min.nc'
L3I_FILE = OUT / 'L3' / f'{BASE}_L3_grid1m_30min_interp.nc'

# close any handles left open by a previous run in this kernel
for _v in ('L1', 'L2', 'L2t', 'L3', 'L3i'):
    if _v in globals():
        try:
            globals()[_v].close()
        except Exception:
            pass

L1  = xr.open_dataset(L1_FILE)
L2  = xr.open_dataset(L2_FILE)
L3  = xr.open_dataset(L3_FILE)            # regular 1 m x 30 min grid (empty bins = NaN)
L3i = xr.open_dataset(L3I_FILE)           # same grid, single empty bins interpolated
# L2 carries cast time as a coord on the 'cast' dim; promote it to a usable axis
L2t = L2.assign_coords(time=L2['time']).swap_dims({'cast': 'time'}).sortby('time')

# --- freshness check: confirm we loaded the files we think we did ---
print('repo:', REPO, '| data:', OUT)
for tag, f in (('L1', L1_FILE), ('L2', L2_FILE), ('L3', L3_FILE), ('L3i', L3I_FILE)):
    st = f.stat()
    print(f'{tag:4s} {st.st_size/1e6:6.0f} MB  modified {_time.ctime(st.st_mtime)}')
print('L1 vars:', list(L1.data_vars))
print('L2 vars:', list(L2.data_vars), '| upcasts:', L2.sizes['cast'])
print('L3 vars:', list(L3.data_vars), '| grid:', dict(L3.sizes))

## 1. Vehicle depth vs time (L1) &mdash; cast-flag check
Single **interactive** plot of the whole deployment: one sample every ~5 s, connected by a thin line so the up/down sawtooth reads as continuous profiles, with markers colored by `cast_direction` (**upcast = red, downcast = blue**). Drag a box to zoom, double-click to reset, pan/save from the toolbar. WebGL (`Scattergl`) — no PNGs written.

In [ ]:
# Single interactive (zoomable) cast-flag check for the whole deployment.
# One point every ~5 s, connected by a line; markers colored by cast direction
# (upcast = red, downcast = blue). Drag a box to zoom; double-click to reset.
import plotly.graph_objects as go

DT_S = 5.0                                  # plot one sample every DT_S seconds
fs   = float(CFG.get('sampling_hz', 2.0))
step = max(1, round(DT_S * fs))             # 2 Hz -> every 10th sample
tt   = L1['time'].values[::step]
dd   = L1['depth'].values[::step]
dr   = L1['cast_direction'].values[::step]
up, dn = dr == 1, dr == 0

fig = go.Figure()
# connecting line (time-ordered) shows the up/down sawtooth
fig.add_scattergl(x=tt, y=dd, mode='lines', line=dict(width=0.6, color='lightgray'),
                  name='depth', hoverinfo='skip', showlegend=False)
fig.add_scattergl(x=tt[dn], y=dd[dn], mode='markers',
                  marker=dict(size=3, color='royalblue'), name='downcast')
fig.add_scattergl(x=tt[up], y=dd[up], mode='markers',
                  marker=dict(size=3, color='crimson'), name='upcast')
fig.update_yaxes(autorange='reversed', title='Depth (m)')      # surface at top
fig.update_xaxes(title='Time')
fig.update_layout(
    title=f'L1 vehicle depth vs time — cast-flag check (1 pt / {DT_S:g} s, {tt.size:,} pts)',
    height=550, dragmode='zoom', legend=dict(itemsizing='constant'),
    margin=dict(l=60, r=20, t=50, b=40))
fig.show()


## 2. Deployment overview &mdash; time vs depth sections (L2 upcasts)

In [ ]:
fields = [('conservative_temperature', 'CT (\u00b0C)', 'thermal'),
          ('practical_salinity',       'Salinity (PSU)', 'haline'),
          ('sigma0',                   '\u03c3\u2080 (kg m\u207b\u00b3)', 'dense')]
try:
    import cmocean
    cmaps = {'thermal': cmocean.cm.thermal, 'haline': cmocean.cm.haline, 'dense': cmocean.cm.dense}
except ImportError:
    cmaps = {'thermal': 'inferno', 'haline': 'viridis', 'dense': 'cividis'}

fig, axes = plt.subplots(len(fields), 1, figsize=(13, 9), sharex=True)
for ax, (var, label, cm) in zip(axes, fields):
    da = L2t[var]
    p = ax.pcolormesh(L2t['time'].values, L2t['depth'].values, da.values,
                      cmap=cmaps[cm], shading='nearest')
    ax.invert_yaxis()
    ax.set_ylabel('Depth (m)')
    fig.colorbar(p, ax=ax, label=label, pad=0.01)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
axes[0].set_title('Wirewalker upcasts \u2014 ' + str(L2.attrs.get('mooring', '')))
fig.tight_layout()

## 2b. Bio-optical & oxygen sections (L2 upcasts)
Time–depth sections of the extra sensors: **log₁₀ chlorophyll**, **log₁₀ backscatter**, and **dissolved O₂ concentration**.

In [ ]:
# L2 upcasts: bio-optical & oxygen sections.
# Chlorophyll and backscatter span orders of magnitude, so they are shown as log10
# (non-positive counts -> NaN); dissolved O2 is shown linearly.
bio = [('chlorophyll',                'log₁₀ Chlorophyll (counts)',          'algae',  True),
       ('backscatter',                'log₁₀ Backscatter (counts)',          'turbid', True),
       ('dissolved_o2_concentration', 'Dissolved O₂ (µmol L⁻¹)',  'oxy',    False)]
bio = [b for b in bio if b[0] in L2t]      # keep only channels present in this deployment

try:
    import cmocean
    cmaps = {'algae': cmocean.cm.algae, 'turbid': cmocean.cm.turbid, 'oxy': cmocean.cm.oxy}
except ImportError:
    cmaps = {'algae': 'viridis', 'turbid': 'cividis', 'oxy': 'magma'}

if not bio:
    print('no bio-optical / oxygen channels in this deployment')
else:
    fig, axes = plt.subplots(len(bio), 1, figsize=(13, 9), sharex=True, squeeze=False)
    for ax, (var, label, cm, logscale) in zip(axes[:, 0], bio):
        da = L2t[var].values
        if logscale:
            with np.errstate(invalid='ignore', divide='ignore'):
                da = np.log10(np.where(da > 0, da, np.nan))
        p = ax.pcolormesh(L2t['time'].values, L2t['depth'].values, da,
                          cmap=cmaps[cm], shading='nearest')
        ax.invert_yaxis()
        ax.set_ylabel('Depth (m)')
        fig.colorbar(p, ax=ax, label=label, pad=0.01)
    axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    axes[0, 0].set_title('Wirewalker upcasts — bio-optical & oxygen — '
                         + str(L2.attrs.get('mooring', '')))
    fig.tight_layout()

## 3. Individual profiles (T, S, sigma vs depth)

In [ ]:
# pick a handful of upcasts spread across the deployment
ncast = L2.sizes['cast']
idx = np.linspace(0, ncast - 1, min(6, ncast)).astype(int)
fig, ax = plt.subplots(1, 3, figsize=(11, 6), sharey=True)
for j in idx:
    c = L2.isel(cast=j)
    lab = str(np.datetime64(c['time'].values, 'h'))
    ax[0].plot(c['conservative_temperature'], L2['depth'], label=lab)
    ax[1].plot(c['practical_salinity'], L2['depth'])
    ax[2].plot(c['sigma0'], L2['depth'])
for a, t in zip(ax, ['CT (\u00b0C)', 'Salinity (PSU)', '\u03c3\u2080 (kg m\u207b\u00b3)']):
    a.set_xlabel(t); a.grid(alpha=0.3)
ax[0].invert_yaxis(); ax[0].set_ylabel('Depth (m)')
ax[0].legend(fontsize=7, title='upcast')
fig.tight_layout()

## 4. Mean profile and stratification

In [ ]:
mean_ct = L2['conservative_temperature'].mean('cast')
mean_sp = L2['practical_salinity'].mean('cast')
mean_sig = L2['sigma0'].mean('cast')
fig, ax = plt.subplots(1, 3, figsize=(11, 6), sharey=True)
ax[0].plot(mean_ct, L2['depth']); ax[0].set_xlabel('mean CT (\u00b0C)')
ax[1].plot(mean_sp, L2['depth']); ax[1].set_xlabel('mean Salinity')
ax[2].plot(mean_sig, L2['depth']); ax[2].set_xlabel('mean \u03c3\u2080')
ax[0].invert_yaxis(); ax[0].set_ylabel('Depth (m)'); [a.grid(alpha=0.3) for a in ax]
fig.suptitle('Deployment-mean upcast profile'); fig.tight_layout()

## 5. L3 regular grid — deployment sections
Depth–time sections (Hovmöller) from the **L3** product (1 m × 30 min regular grid, Nyquist 1 cph). Empty 30-min bins (~8%, from the cadence/bin phase slip) show as gaps. Writes `L3_sections.png`.

In [ ]:
# L3 regular grid (1 m x 30 min): full-deployment depth-time sections.
fields = [('conservative_temperature', 'CT (°C)', 'thermal'),
          ('practical_salinity',       'Salinity (PSU)', 'haline'),
          ('sigma0',                   'σ₀ (kg m⁻³)', 'dense')]
try:
    import cmocean
    cmaps = {'thermal': cmocean.cm.thermal, 'haline': cmocean.cm.haline, 'dense': cmocean.cm.dense}
except ImportError:
    cmaps = {'thermal': 'inferno', 'haline': 'viridis', 'dense': 'cividis'}

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, (var, label, cm) in zip(axes, fields):
    p = ax.pcolormesh(L3['time'].values, L3['depth'].values, L3[var].values,
                      cmap=cmaps[cm], shading='nearest', rasterized=True)
    ax.invert_yaxis(); ax.set_ylabel('Depth (m)')
    fig.colorbar(p, ax=ax, label=label, pad=0.01)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[0].set_title('L3 regular grid (1 m × 30 min) — NOPP-Aleutians Wirewalker')
fig.tight_layout()
fig.savefig(FIGS / 'L3_sections.png', dpi=130); print('wrote', FIGS / 'L3_sections.png')

## 5b. L3 interpolated — deployment sections
Same sections from the **gap-filled L3 companion** (`L3i`): single empty 30-min bins linearly interpolated (≈8% → ~1% gaps), so the phase-slip stripes are gone. Real multi-hour gaps remain. Writes `L3_sections_interp.png`.

In [ ]:
# L3 interpolated companion: same sections, single empty bins gap-filled (cleaner).
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, (var, label, cm) in zip(axes, fields):
    p = ax.pcolormesh(L3i['time'].values, L3i['depth'].values, L3i[var].values,
                      cmap=cmaps[cm], shading='nearest', rasterized=True)
    ax.invert_yaxis(); ax.set_ylabel('Depth (m)')
    fig.colorbar(p, ax=ax, label=label, pad=0.01)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[0].set_title('L3 interpolated (1 m × 30 min, single gaps filled) — NOPP-Aleutians Wirewalker')
fig.tight_layout()
fig.savefig(FIGS / 'L3_sections_interp.png', dpi=130); print('wrote', FIGS / 'L3_sections_interp.png')